# Transform Customers Data
1. Remove records with NULL customer_id
2. Remove exact duplicate records
3. Remove duplicate records based on created_timestamp
4. CAST the column to correct Data Type 
5. Write Transformed data to Silver schema


# 1. Remove records with NULL customer_id

In [0]:
%python
from pyspark.sql import functions as f
from pyspark.sql.types import *

silver_customer_df = spark.table("gizmobox.bronze.v_customers")
silver_customer_df = silver_customer_df.filter(f.col("customer_id").isNotNull())
silver_customer_df.show()

#2. Remove exact duplicate records

In [0]:
%python
dedup_silver_customer_df = silver_customer_df.distinct()
dedup_silver_customer_df.show()

#3. Remove duplicate records based on created_timestamp

In [0]:
%python
from pyspark.sql.window import Window
from pyspark.sql.functions import *

window = Window.partitionBy("customer_id").orderBy(f.col("created_timestamp").desc())
dedup_silver_customer_df = dedup_silver_customer_df.withColumn("rn", row_number().over(window))
dedup_silver_customer_df = dedup_silver_customer_df.filter(f.col("rn") == 1)
dedup_silver_customer_df = dedup_silver_customer_df.drop("rn")
dedup_silver_customer_df.show()

#4. CAST the column to correct Data Type


In [0]:
%python
dedup_silver_customer_df.printSchema()

In [0]:
%python
dedup_silver_customer_df = dedup_silver_customer_df.select(f.col("file_path").cast("string").alias("file_path"),
                                                    f.col("created_timestamp").cast("timestamp").alias("created_timestamp"),
                                                    f.col("customer_id").cast("int").alias("customer_id"),
                                                    f.col("customer_name").cast("string").alias("customer_name"),
                                                    f.col("date_of_birth").cast("date").alias("date_of_birth"),
                                                    f.col("email").cast("string").alias("email"),
                                                    f.col("member_since").cast("date").alias("member_since"),
                                                    f.col("telephone").cast("string").alias("telephone")
                                                    )
dedup_silver_customer_df.show(truncate=0, n=10000)

#5. Write Transformed data to Silver schema
## Write Data to Delta Table

In [0]:
%python
dedup_silver_customer_df.count()

In [0]:
%python
dedup_silver_customer_df = dedup_silver_customer_df.withColumn("created_timestamp", dedup_silver_customer_df["created_timestamp"].cast("timestamp")); dedup_silver_customer_df.write.mode("overwrite").saveAsTable("gizmobox.silver.customers_silver")

In [0]:
select * from gizmobox.silver.customers_silver;

In [0]:
describe extended gizmobox.silver.customers_silver;